In [0]:
# List files in the landing directory
ecommerce_path_orders = "/Volumes/business_lakehouse/landing/raw_volume/ecommerce/orders"
ecommerce_path_products = "/Volumes/business_lakehouse/landing/raw_volume/ecommerce/products"
ecommerce_path_users = "/Volumes/business_lakehouse/landing/raw_volume/ecommerce/users"
exchange_rates_path = "/Volumes/business_lakehouse/landing/raw_volume/exchange_rates"

ecommerce_files_orders = dbutils.fs.ls(ecommerce_path_orders)
ecommerce_files_products = dbutils.fs.ls(ecommerce_path_products)
ecommerce_files_users = dbutils.fs.ls(ecommerce_path_users)
rates_files = dbutils.fs.ls(exchange_rates_path)

display(ecommerce_files_orders)
display(ecommerce_files_products)
display(ecommerce_files_users)
display(rates_files)

In [0]:
# Read orders JSON into a Spark DataFrame

from pyspark.sql.functions import col, size

orders_path = ecommerce_files_orders[1].path

df_orders = spark.read.json(orders_path)

# Inspect DataFrame schema
df_orders.printSchema()

# Display number of orders for first 10 cases
df_orders.select(
    'year',
    'month',
    'total',
    size(col('orders')).alias('orders_count')
    ).show(10)

In [0]:
# Read products and users json files

products_path = ecommerce_files_products[0].path
users_path = ecommerce_files_users[0].path

df_products = spark.read.json(products_path)
df_users = spark.read.json(users_path)

# Inspect DataFrame schema
df_products.printSchema()
df_users.printSchema()

# Display number of products and users
df_products.count()
df_users.count()

# Display sample records
display(df_products.limit(5))
display(df_users.limit(5))

In [0]:
# Inspect the beginning of the XML file

from xml.dom import minidom

xml_path = "/Volumes/business_lakehouse/landing/raw_volume/exchange_rates/eurofxref-hist.xml"

# Read the beginning of the XML file
xml_content = dbutils.fs.head(xml_path, 10000)

# Extract the first daily exchange rate block
start = xml_content.find('<Cube time=')
end = xml_content.find('</Cube>', start) + len('</Cube>')

daily_xml = xml_content[start:end]

# Format XML
xml_doc = minidom.parseString(daily_xml)

print(xml_doc.toprettyxml(indent="    "))

In [0]:
df_rates = (
    spark.read
    .format('xml')
    .option('rowTag', 'Cube')
    .option('inferSchema', 'true')
    .load(xml_path)
)

In [0]:
print("Number of rows:", df_rates.count())

df_rates.selectExpr(
    "size(Cube) AS number_of_days"
).show(10)

In [0]:
SOURCE_PATH = "/Volumes/business_lakehouse/landing/raw_volume/ecommerce/products"

SCHEMA_PATH = (
    "/Volumes/business_lakehouse/landing/raw_volume/"
    "_schemas/products_typed"
)

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_PATH)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(SOURCE_PATH)
)

print(df.isStreaming)
df.printSchema()

In [0]:
%sql
SELECT
    COUNT(*) AS row_count,
    SUM(size(products)) AS products_count
FROM business_lakehouse.bronze.products_incremental;

In [0]:
import json

new_products = {
    "limit": 2,
    "skip": 194,
    "total": 196,
    "products": [
        {
            "id": 195,
            "title": "Wireless Mechanical Keyboard",
            "category": "laptops",
            "brand": "TechPro",
            "price": 89.99,
            "stock": 120
        },
        {
            "id": 196,
            "title": "USB-C Docking Station",
            "category": "laptops",
            "brand": "TechPro",
            "price": 129.99,
            "stock": 75
        }
    ]
}

file_path = (
    "/Volumes/business_lakehouse/landing/raw_volume/"
    "ecommerce/products/products_new.json"
)

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(new_products, f, indent=4)

print("File created:", file_path)